In [1]:
import pandas as pd
import numpy as np
import joblib

# build features using the same logic as notebook 03
movies = pd.read_csv("../data/ml-32m/movies.csv") # movieId,title,genres
links = pd.read_csv("../data/ml-32m/links.csv")

# use filtered_ratings2.csv for testing, which contains userId, tmdbId, rating, timestamp
# from user_id retrieve respective tmdb_id
ratings = pd.read_csv("../data/filtered_ratings2.csv")

# process ratings so that it is per user

movie_embeddings = pd.read_parquet("../data/movie_embeddings.parquet") # tmdbId, embedding_0, embedding_1, ..., embedding_n

# assume tmdb_id and respective rating is given to build user features for the userNN model
# per user
def build_user_features(ratings, movie_embeddings):
    """
    ratings (per user ratings) - DataFrame containing tmdbId, rating, timestamp
    movie_embeddings - DataFrame containing tmdbId and ALL movie embeddings
    """

    users_encoded = pd.DataFrame(index=[0])

    # genre avg ratings
    genre_columns = [
        "Action",
        "Adventure",
        "Animation",
        "Children",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "IMAX",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ]
    # export genre columns
    pd.Series(genre_columns, name="genre").to_csv(
    "../data/genre_columns.csv",
    index=False
    )

    for genre in genre_columns:
        genre_movie_ids = movie_embeddings.index[movie_embeddings[f"movie_{genre}"] == 1]
        users_encoded[genre] = ratings[ratings.index.isin(genre_movie_ids)]["rating"].mean()

    # decade avg ratings
    decade_columns = [col for col in movie_embeddings.columns if col.startswith("movie_Decade_")]
    for decade in decade_columns:
        decade_movie_ids = movie_embeddings.index[movie_embeddings[decade] == 1]
        user_decade = decade.removeprefix("movie_")
        users_encoded[user_decade] = ratings[ratings.index.isin(decade_movie_ids)]["rating"].mean()

    users_encoded["avg_rating"] = ratings["rating"].mean()
    users_encoded["std_rating"] = ratings["rating"].std()
    users_encoded["count_rating"] = ratings["rating"].count()

    users_encoded = users_encoded.fillna(0)

    user_feature_frame = users_encoded.rename(
        columns={col: f"user_{col}" for col in users_encoded.columns}
    )

    return user_feature_frame

# displaying multiple users in one frame
users_feature_frame2 = pd.DataFrame()
for user_id in ratings["userId"].unique():
    user_ratings = ratings[ratings["userId"] == user_id]
    user_feature_frame = build_user_features(user_ratings, movie_embeddings)
    user_feature_frame["userId"] = user_id
    users_feature_frame2 = pd.concat([users_feature_frame2, user_feature_frame], ignore_index=True)

#userid should be set as index for the user embeddings dataframe - tbc 
users_feature_frame2.set_index("userId", inplace=True)

print("User feature frame:", users_feature_frame2.shape)
print(users_feature_frame2.head())
print("User feature columns:", users_feature_frame2.columns.tolist())

# export user embeddings to parquet to be used in the hybrid model
users_feature_frame2.to_parquet("../data/users_feature_frame2.parquet", index=True)



User feature frame: (1000, 36)
        user_Action  user_Adventure  user_Animation  user_Children  \
userId                                                               
96         4.178571        4.062500        4.000000       4.000000   
218        4.112903        4.145833        3.625000       3.000000   
282        4.583333        4.500000        0.000000       4.833333   
468        3.564103        3.774194        3.857143       4.200000   
704        3.906977        3.775510        3.125000       3.090909   

        user_Comedy  user_Crime  user_Documentary  user_Drama  user_Fantasy  \
userId                                                                        
96         4.052632    4.352941             4.000    4.239130      3.833333   
218        3.919355    3.961538             3.900    3.700000      3.714286   
282        4.500000    4.750000             4.125    4.517857      4.285714   
468        3.673913    3.550000             3.000    3.647541      3.812500   
704 

In [2]:
user_scaler = joblib.load('../models/scaler_users.pkl')
user_feature_scaled = user_scaler.transform(users_feature_frame2)

In [3]:
from tensorflow.keras.models import load_model

userNN = load_model('../models/userNN.keras')

user_embeddings = userNN.predict(user_feature_scaled)

user_embeddings_df = pd.DataFrame(user_embeddings, index=users_feature_frame2.index)

user_embeddings_df.to_parquet("../data/user_embeddings_nn.parquet", index=True)

print("User embedding shape:", user_embeddings_df.shape)
print(user_embeddings_df.head())

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
User embedding shape: (1000, 32)
                   0            1              2              3   \
userId                                                             
96      147366.453125  2212562.250 -343158.843750  760282.562500   
218    -228689.500000  2792056.250 -281603.812500  174551.031250   
282      27811.439453  2788385.000 -257908.546875  676959.375000   
468      34599.230469  1408727.125 -532550.687500  322695.968750   
704    -149367.718750   410948.375  158016.406250  -76266.742188   

                  4             5            6            7              8   \
userId                                                                        
96      691862.31250 -5.611378e+05 -201660.5000  2483677.750  -23224.804688   
218     633624.12500 -1.226066e+06 -531707.3125  2929054.750   78480.843750   
282     691363.81250 -1.050819e+06 -685335.1250  2981586.000  101991.804688   
468     470377.96875 -9.251531e+05 -360882.0625  131144